# 07 - Extended data, 2015-2025

**Goal:** replace the three Kaggle years with every complete year from 2015 to 2025, from official Seoul files, and show that the new series is consistent and matches the official figures.

Notebooks 00-06 still use the Kaggle files (2017-2019); this notebook builds `data/processed/air_hourly_2015_2025.parquet` and `air_daily_2015_2025.parquet`, which later analyses use. The code lives in `src/seoul_air/sources.py`.

## 1. Imports

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from seoul_air import config
import contextlib
import io
import json

from seoul_air import sources
from seoul_air.config import POLLUTANTS
from seoul_air.data import (complete_hourly_grid, daily_means, keep_valid_readings, load_measurements,
                            to_wide)
from seoul_air.grades import pm_grade
from seoul_air.report import save_key_figures
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 140)
config.FIGURES.mkdir(parents=True, exist_ok=True)
config.METRICS.mkdir(parents=True, exist_ok=True)

## 2. Candidate sources

Three official sources hold hourly readings of the 25 Seoul urban stations. The table compares their coverage, format, hour convention, instrument status, whether the data is final, and licence.

In [2]:
pd.set_option("display.max_colwidth", 70)
sources.SOURCE_CATALOG.set_index("source").drop(columns="url").T

source,Kaggle bappekim/air-pollution-in-seoul,OA-15526 Seoul air pollution measurements,OA-2221 Seoul hourly air quality by period,AirKorea final measurements (data.go.kr 15122830)
publisher,"Kaggle user, from OA-15526",Seoul Research Institute of Public Health and Environment,Seoul Air Quality Policy Division,Korea Environment Corporation / NIER
years_available,2017-2019,1988-2022,2019-2025,versions 2018-2024 on data.go.kr
used_for,notebooks 00-06,2015-2022,2023-2025,check of 2024 (tools/compare_airkorea_final.py)
format,"CSV, long, one file","CSV per year, long, UTF-8","CSV per month (CP949) to 2020-06, ZIP per year (UTF-8) after; wide","XLSX per year, all Korean stations, about 400 MB"
hour_label,start of hour (00-23),start of hour (00-23),end of hour (00 = 24 of the day before),end of hour (01-24)
instrument_status,yes,yes,no (flagged readings left blank),no (quality-assured values only)
final,no (copy of provisional data),no (stated on the page),no (same values as OA-15526),yes
licence,CC BY-SA 4.0,KOGL Type 1 (attribution),KOGL Type 1 (attribution),"KOGL Type 3 (attribution, no derivatives)"


## 3. Download

`download_extended` fetches every yearly file from the Seoul Open Data Plaza and checks the file name the server returns, so a change of the portal's file ids stops the download instead of saving a wrong year. No file needs to be downloaded by hand. The AirKorea final data is not part of the pipeline: it is only used for the check of section 4.4.

In [3]:
paths = sources.download_extended()
inventory = []
for path in paths:
    frame = sources.read_oa15526(path) if path.suffix == ".csv" else sources.read_oa2221(path)
    inventory.append({"file": path.name, "MB": round(path.stat().st_size / 1e6, 1), "readings": len(frame),
                      "first_hour": frame["datetime"].min(), "last_hour": frame["datetime"].max(),
                      "stations": frame["station_code"].nunique()})
pd.DataFrame(inventory)

,file,MB,readings,first_hour,last_hour,stations
0,AIR_HOUR_2015.csv,38.5,1313712,2015-01-01 00:00:00,2015-12-31 23:00:00,25
1,AIR_HOUR_2016.csv,38.8,1317600,2016-01-01 00:00:00,2016-12-31 23:00:00,25
2,AIR_HOUR_2017.csv,38.5,1314000,2017-01-01 00:00:00,2017-12-31 23:00:00,25
3,AIR_HOUR_2018.csv,38.4,1314000,2018-01-01 00:00:00,2018-12-31 23:00:00,25
4,AIR_HOUR_2019.csv,38.5,1314000,2019-01-01 00:00:00,2019-12-31 23:00:00,25
5,AIR_HOUR_2020.csv,38.5,1317588,2020-01-01 00:00:00,2020-12-31 23:00:00,25
6,AIR_HOUR_2021.csv,38.4,1314000,2021-01-01 00:00:00,2021-12-31 23:00:00,25
7,AIR_HOUR_2022.csv,38.4,1314000,2022-01-01 00:00:00,2022-12-31 23:00:00,25
8,OA-2221_2021.zip,2.2,1288522,2020-12-31 23:00:00,2021-12-31 22:00:00,25
9,OA-2221_2022.zip,2.2,1290479,2021-12-31 23:00:00,2022-12-31 22:00:00,25


**Interpretation.** All 13 files hold the 25 urban stations. Each OA-15526 file covers one calendar year by the start of the hour (2015 has 1,313,712 readings). Each OA-2221 file starts at 23:00 of the year before once its labels are moved to the start of the hour, because its first label, 00:00 of 1 January, closes the last hour of the previous year; the loader therefore reads the OA-2221 files together and assigns every hour to its own year.

## 4. Are the sources consistent?

`agreement` moves the timestamps of the second source by -1, 0 and +1 hour and counts identical readings. Timestamps of both sources are first brought to this project's convention (start of the hour, see notebook 01).

### 4.1 Kaggle and OA-15526 (2019)

In [4]:
kaggle = load_measurements()
kaggle = kaggle[(kaggle["datetime"].dt.year == 2019) & (kaggle["status"] == config.STATUS_NORMAL)]
oa15526_2019 = sources.read_oa15526(sources.raw_path("OA-15526", 2019))
oa15526_2019 = oa15526_2019[oa15526_2019["status"] == config.STATUS_NORMAL]
sources.agreement(oa15526_2019, kaggle)

,lag_hours,compared,identical,identical_pct
0,-1,1201350,368824,30.700795
1,0,1208437,1205434,99.751497
2,1,1201374,368830,30.700681


### 4.2 OA-2221 and OA-15526 (2021-2022), after moving OA-2221 to the start of the hour

In [5]:
rows = []
for year in [2021, 2022]:
    a = sources.read_oa15526(sources.raw_path("OA-15526", year))
    a = a[(a["status"] == config.STATUS_NORMAL) & (a["datetime"].dt.year == year)]
    b = sources.read_oa2221(sources.raw_path("OA-2221", year))
    rows.append(sources.agreement(a, b).assign(year=year))
overlap = pd.concat(rows).set_index(["year", "lag_hours"])
overlap

compared  identical  identical_pct
year lag_hours                                    
2021 -1          1281942     428277      33.408454
      0          1288076    1288013      99.995109
      1          1282263     428378      33.407967
2022 -1          1283188     435076      33.905866
      0          1289201    1289201     100.000000
      1          1283473     435188      33.907063

In [6]:
# Readings that OA-15526 flags: are they in OA-2221?
a = sources.read_oa15526(sources.raw_path("OA-15526", 2022))
b = sources.read_oa2221(sources.raw_path("OA-2221", 2022))
keys = ["datetime", "station_code", "pollutant"]
merged = a.astype({"pollutant": str}).merge(b[keys + ["value"]].astype({"pollutant": str}), on=keys,
                                             how="left", suffixes=("", "_2221"))
flags = merged.groupby(merged["status"].map(config.STATUS_LABELS)).agg(
    readings=("value", "size"), kept_in_OA2221_pct=("value_2221", lambda s: 100 * s.notna().mean()))
flags.round(2)

,readings,kept_in_OA2221_pct
status,,
Abnormal,605,0.00
Abnormal data,5803,19.32
Needs calibration,11674,0.00
Normal,1289671,99.96
Power cut off,1214,0.00
Under repair,5033,0.16


In [7]:
# The flagged readings that OA-2221 keeps, and their effect on the 2022 annual means
kept = merged[(merged["status"] != config.STATUS_NORMAL) & merged["value_2221"].notna()]
print("Flagged readings kept in OA-2221 (2022):", len(kept), kept.groupby("pollutant").size().to_dict())
print(kept.groupby("pollutant")["value"].describe()[["count", "min", "50%", "max"]].round(3).to_string())
effect = {}
for pollutant in ["pm25", "pm10"]:
    rows = merged[merged["pollutant"] == pollutant]
    normal_only = rows[rows["status"] == config.STATUS_NORMAL].groupby("station_code")["value"].mean()
    oa2221 = rows.groupby("station_code")["value_2221"].mean()
    effect[pollutant] = {"largest station difference": (normal_only - oa2221).abs().max(),
                         "city difference": normal_only.mean() - oa2221.mean()}
pd.DataFrame(effect).T.round(4)

Flagged readings kept in OA-2221 (2022): 1129 {'co': 540, 'no2': 109, 'o3': 445, 'pm10': 27, 'so2': 8}
           count    min    50%    max
pollutant                            
co         540.0  0.100  0.200  1.000
no2        109.0  0.003  0.010  0.047
o3         445.0  0.002  0.036  0.073
pm10        27.0  3.000  3.000  3.000
so2          8.0  0.003  0.004  0.005


,largest station difference,city difference
pm25,0.0653,0.0004
pm10,0.0742,0.0037


**Interpretation.** **OA-15526 and Kaggle are the same readings:** 99.75% are identical at the same hour against 30.7% with a one-hour shift, so the Kaggle file is a copy of OA-15526 with the same start-of-hour labels. **OA-2221 holds the same readings as OA-15526** once its labels are moved back one hour: 99.995% identical in 2021 and 100% in 2022. **Flagged readings:** OA-2221 keeps 99.96% of the normal readings and none of those flagged as abnormal, needing calibration or with the power cut off, but in 2022 it keeps 19.32% of the readings flagged as abnormal data and 0.16% of those under repair: 1,129 readings, mostly CO and O3, none of PM2.5, and 27 of PM10 that all equal 3. Their effect on the 2022 annual means is at most 0.0742 ug/m3 for a station and 0.0037 for the city. OA-2221 has no status column, so similar readings in 2023-2025 cannot be removed; this check suggests that their effect is negligible.

### 4.3 Provisional and final data (2024)

Both Seoul sources are provisional. The AirKorea final data of 2024 was compared with OA-2221 2024 by `tools/compare_airkorea_final.py` (it downloads a workbook of about 400 MB and needs `python-calamine`, so it is not run here); its summary is stored in `data/reference/final_vs_provisional_2024.csv`.

In [8]:
final_check = pd.read_csv(config.ROOT / "data" / "reference" / "final_vs_provisional_2024.csv")
final_check.drop(columns=["year", "final_source"]).set_index("pollutant").round(4)

,hours,final_present_pct,provisional_present_pct,identical_where_both_pct,only_provisional_pct,only_final_pct,city_annual_mean_final,city_annual_mean_provisional,max_station_annual_mean_diff
pollutant,,,,,,,,,
co,219600,97.6885,97.8260,99.2879,0.1790,0.0414,0.4125,0.4129,0.0056
no2,219600,98.0624,98.1293,99.2810,0.0820,0.0150,0.0188,0.0188,0.0001
o3,219600,98.0346,98.0200,99.2702,0.0005,0.0150,0.0332,0.0332,0.0000
pm10,219600,97.6034,97.5783,100.0000,0.0205,0.0455,31.3183,31.3207,0.0203
pm25,219600,98.1252,98.0883,100.0000,0.0205,0.0574,17.5505,17.5542,0.0247
so2,219600,97.9362,97.9440,99.2908,0.0246,0.0168,0.0027,0.0027,0.0000


**Interpretation.** For particulate matter the provisional readings of 2024 equal the final ones: where both have a value, 100% of the PM10 and PM2.5 readings are identical, and 99.27% to 99.29% of the gas readings. The final data adds or removes between 0.0005% and 0.179% of the hours. The 2024 city PM2.5 mean is 17.5505 ug/m3 with the final data and 17.5542 with the provisional data, and no station differs by more than 0.0247. Only 2024 was checked; `tools/compare_airkorea_final.py` can check the other years with a final workbook on data.go.kr once their record keys are added.

## 5. Build the 2015-2025 tables

Year by year, to keep memory low: OA-15526 for 2015-2022 and OA-2221 for 2023-2025, readings with a non-normal status dropped, then the complete hourly grid and the daily means with the rules of notebook 01 (at least 18 valid hours).

In [9]:
stations = sources.load_stations_reference()
pollutants = list(POLLUTANTS)
wide_years, cleaning = [], []
for year in range(2015, 2026):
    long = sources.load_extended_measurements(year, year)
    with contextlib.redirect_stdout(io.StringIO()):
        valid = keep_valid_readings(long)
    cleaning.append({"year": year, "source": long["source"].iloc[0], "readings": len(long),
                     "dropped_status": len(long) - len(valid),
                     "dropped_pct": 100 * (len(long) - len(valid)) / len(long)})
    wide_years.append(to_wide(valid))
    del long, valid
cleaning = pd.DataFrame(cleaning).set_index("year")
cleaning.round(2)

,source,readings,dropped_status,dropped_pct
year,,,,
2015,OA-15526,1313712,42165,3.21
2016,OA-15526,1317600,41615,3.16
2017,OA-15526,1314000,26712,2.03
2018,OA-15526,1314000,39715,3.02
2019,OA-15526,1314000,47739,3.63
2020,OA-15526,1317588,21971,1.67
2021,OA-15526,1314000,25741,1.96
2022,OA-15526,1314000,24329,1.85
2023,OA-2221,1279137,0,0.00


In [10]:
wide = pd.concat(wide_years, ignore_index=True)
del wide_years
hourly = complete_hourly_grid(wide, "2015-01-01 00:00", "2025-12-31 23:00", sorted(stations["seoul_code"]))
hourly = hourly.merge(stations[["seoul_code", "district"]], left_on="station_code", right_on="seoul_code").drop(columns="seoul_code")
hourly = hourly[["datetime", "station_code", "district"] + pollutants]
daily = daily_means(hourly, pollutants)
daily = daily.merge(stations[["seoul_code", "district"]], left_on="station_code", right_on="seoul_code").drop(columns="seoul_code")
daily = daily[["date", "station_code", "district"] + pollutants]
daily["pm25_grade"] = pm_grade(daily["pm25"].round(), "pm25")
daily["pm10_grade"] = pm_grade(daily["pm10"].round(), "pm10")
hourly.to_parquet(config.DATA_PROCESSED / "air_hourly_2015_2025.parquet", index=False)
daily.to_parquet(config.DATA_PROCESSED / "air_daily_2015_2025.parquet", index=False)
print("Hourly grid:", hourly.shape, "| daily:", daily.shape)

city_days = daily.groupby("date")["pm25"].count()
gaps = city_days[city_days == 0]
coverage = pd.DataFrame({
    "missing_hours_pm25_pct": hourly.groupby(hourly["datetime"].dt.year)["pm25"].apply(lambda s: 100 * s.isna().mean()),
    "days_without_city_value": gaps.groupby(gaps.index.year).size(),
}).fillna(0)
coverage.index.name = "year"
coverage.round(2)

Hourly grid: (2410800, 9) | daily: (100450, 11)


,missing_hours_pm25_pct,days_without_city_value
year,,
2015,4.77,0.0
2016,3.43,0.0
2017,2.82,0.0
2018,3.04,0.0
2019,3.79,0.0
2020,1.59,0.0
2021,1.75,0.0
2022,1.92,0.0
2023,2.07,0.0


In [11]:
# The 18 days that the Kaggle file lacks in 2019 (notebook 01)
kaggle_gaps = pd.read_csv(config.METRICS / "01_network_gap_days.csv", parse_dates=["date"])
on_gap_days = daily[daily["date"].isin(kaggle_gaps["date"])]
print("Kaggle gap days:", len(kaggle_gaps), "| station-days with a valid PM2.5 mean on them now:",
      int(on_gap_days["pm25"].notna().sum()), "of", len(on_gap_days))

Kaggle gap days: 18 | station-days with a valid PM2.5 mean on them now: 437 of 450


**Interpretation.** The status filter drops between 1.67% and 3.63% of the readings of each year in 2015-2022; nothing is dropped in 2023-2025, where OA-2221 has already left flagged readings blank. Between 1.59% and 4.77% of the PM2.5 hours are missing each year, and **no day lacks a value for the city in any year**. This corrects notebook 01: on the 18 days of 2019 without data in the Kaggle file, 437 of the 450 station-days now have a valid PM2.5 mean. Those days are missing from the Kaggle copy, not from the network. The table has one row per station and hour from 2015 to 2025; the last hour of 2025 is empty because it belongs to the 2026 file.

## 6. Validation against the official annual means

`data/reference/official_annual_means.csv` holds two kinds of official figures:

- **Station level:** the annual PM10 and PM2.5 means of three urban stations (Gangnam-gu, Gangdong-gu, Gangbuk-gu) for every year, from the Seoul Open API (dataset OA-2216). Without a personal key the API returns only the first 5 rows of each year, which include these three stations. The official values are whole numbers.
- **City level:** the whole-Seoul PM2.5 (and one PM10) annual means that the Seoul Metropolitan Government announced, as quoted in the press.

The project's annual mean of a station is the mean of its valid hourly readings; the city value is the mean of the 25 station means.

In [12]:
official = pd.read_csv(config.ROOT / "data" / "reference" / "official_annual_means.csv")
hourly["year"] = hourly["datetime"].dt.year
station_year = (hourly.groupby(["year", "district"])[["pm10", "pm25"]].mean()
                .reset_index().melt(id_vars=["year", "district"], var_name="pollutant", value_name="project"))
station_check = official[official["scope"] == "station"].merge(station_year, on=["year", "district", "pollutant"])
station_check["project_rounded"] = station_check["project"].round()
station_check["difference"] = station_check["project_rounded"] - station_check["official_value"]
print(f"Station-years compared: {len(station_check)} | identical after rounding: "
      f"{(station_check['difference'] == 0).mean():.1%} | within 1 ug/m3: {(station_check['difference'].abs() <= 1).mean():.1%}")
station_check.pivot_table(index="year", columns=["district", "pollutant"], values="difference")

Station-years compared: 66 | identical after rounding: 86.4% | within 1 ug/m3: 98.5%


district  Gangbuk-gu      Gangdong-gu      Gangnam-gu     
pollutant       pm10 pm25        pm10 pm25       pm10 pm25
year                                                      
2015             2.0  1.0         0.0  0.0        0.0  0.0
2016             0.0  0.0         0.0  0.0        0.0  0.0
2017             0.0  0.0         0.0  0.0        0.0  0.0
2018             0.0  0.0         0.0  0.0        0.0  0.0
2019            -1.0  0.0        -1.0  0.0       -1.0  0.0
2020             0.0  0.0         0.0  0.0        0.0  0.0
2021             1.0  0.0        -1.0  0.0        0.0  0.0
2022             0.0  0.0         0.0  0.0        0.0  0.0
2023             0.0  0.0        -1.0 -1.0        0.0  0.0
2024             0.0  0.0         0.0  0.0        0.0  0.0
2025             0.0  0.0         0.0  0.0        0.0  0.0

In [13]:
city_year = station_year.groupby(["year", "pollutant"])["project"].mean()
city_check = official[official["scope"] == "city"][["year", "pollutant", "official_value", "url"]].copy()
city_check["project"] = [city_year.loc[(y, p)] for y, p in zip(city_check["year"], city_check["pollutant"])]
decimals = city_check["official_value"].map(lambda v: 0 if float(v).is_integer() else 1)
city_check["project_rounded_like_official"] = [round(v, d) for v, d in zip(city_check["project"], decimals)]
city_check["difference"] = city_check["project"] - city_check["official_value"]
validation = pd.concat([station_check.assign(scope="station"), city_check.assign(scope="city")], ignore_index=True)
validation.to_csv(config.METRICS / "07_validation.csv", index=False)
city_check.drop(columns="url").round(2)

,year,pollutant,official_value,project,project_rounded_like_official,difference
66,2015,pm25,23.0,23.29,23.0,0.29
67,2020,pm25,21.0,20.70,21.0,-0.30
68,2021,pm25,19.8,19.84,19.8,0.04
69,2024,pm25,17.6,17.55,17.6,-0.05
70,2025,pm25,18.0,18.47,18.0,0.47
71,2025,pm10,32.0,31.87,32.0,-0.13


In [14]:
print("Project annual means for Seoul (mean of the 25 stations, ug/m3):")
city_year.unstack().round(2)

Project annual means for Seoul (mean of the 25 stations, ug/m3):


pollutant,pm10,pm25
year,,
2015,45.50,23.29
2016,47.83,26.20
2017,43.80,24.60
2018,39.69,22.79
2019,42.11,24.98
2020,35.30,20.70
2021,38.25,19.84
2022,32.86,18.38
2023,37.85,19.68


**Interpretation.** **Station level:** of 66 station-years, 86.4% equal the official value once rounded to whole numbers as the official figures are, and 98.5% are within 1 ug/m3. The one larger difference is the PM10 of Gangbuk-gu in 2015 (2 ug/m3). Differences of one unit are expected: the official figures come from the final data and are rounded. **City level:** the project gives 23.29, 20.70, 19.84, 17.55 and 18.47 ug/m3 of PM2.5 for 2015, 2020, 2021, 2024 and 2025, against official figures of 23, 21, 19.8, 17.6 and 18, and 31.87 against 32 for PM10 in 2025: every value equals the official one when rounded to the same precision. The method of the official city figure is not published; the mean of the 25 station means reproduces it. The extended series is therefore consistent with the official statistics. Over 2015-2025 the city PM2.5 mean goes from 26.20 ug/m3 in 2016, its highest, to 17.55 in 2024, its lowest; the next phase of the analysis studies this trend (docs/ANALYSIS_PLAN.md).

## 7. Changes of methodology

| Topic | Notebooks 00-06 (Kaggle) | Extended data (this notebook) |
|---|---|---|
| Source | Kaggle copy of OA-15526, 2017-2019 | OA-15526 2015-2022, OA-2221 2023-2025 |
| Hour label | Start of hour | Start of hour; OA-2221 moved back one hour (it labels the end) |
| Faulty readings | Dropped by instrument status | Same for 2015-2022; for 2023-2025 the provider already leaves them blank (no status column) |
| Data status | Provisional | Provisional; checked against the final data of 2024 and the official annual means |
| Station codes | Seoul codes (101-125) | Same; OA-2221 uses AirKorea codes, mapped with `data/reference/stations.csv` |
| Stations | One location each (latest) | Same limitation; the station history covers changes up to 2019 only |
| Year boundaries | - | The last hour of 2025 sits in the 2026 file and is missing |
| 24-hour PM10 | - | The rolling 24-hour PM10 of OA-2221 is not used (it is derived from the hourly values) |

## 8. Key figures

In [15]:
save_key_figures("07", {
    "years": [2015, 2025],
    "station_years_compared": len(station_check),
    "station_identical_pct": 100 * (station_check["difference"] == 0).mean(),
    "station_within_1_pct": 100 * (station_check["difference"].abs() <= 1).mean(),
    "city_pm25_by_year": city_year.xs("pm25", level="pollutant").to_dict(),
    "city_official": {f"{y}_{p}": v for y, p, v in zip(city_check["year"], city_check["pollutant"], city_check["official_value"])},
    "final_vs_provisional_2024_pm25_city_diff": float(
        final_check.set_index("pollutant").loc["pm25", "city_annual_mean_final"]
        - final_check.set_index("pollutant").loc["pm25", "city_annual_mean_provisional"]),
})
print((config.METRICS / "07_key_figures.json").read_text())

{
  "city_official": {
    "2015_pm25": 23.0,
    "2020_pm25": 21.0,
    "2021_pm25": 19.8,
    "2024_pm25": 17.6,
    "2025_pm10": 32.0,
    "2025_pm25": 18.0
  },
  "city_pm25_by_year": {
    "2015": 23.291244705584845,
    "2016": 26.198156610257964,
    "2017": 24.60434660411735,
    "2018": 22.79473597821834,
    "2019": 24.97685813717377,
    "2020": 20.701766401956164,
    "2021": 19.84229932225568,
    "2022": 18.380780079453597,
    "2023": 19.681735950101057,
    "2024": 17.553596804518182,
    "2025": 18.47052830407689
  },
  "final_vs_provisional_2024_pm25_city_diff": -0.0036670229579058855,
  "station_identical_pct": 86.36363636363636,
  "station_within_1_pct": 98.48484848484848,
  "station_years_compared": 66,
  "years": [
    2015,
    2025
  ]
}

